## Dense Neural Network Grid Search

In [14]:
# Import libraries
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import tensorflow as tf

from tensorflow.keras import layers, models, callbacks, regularizers

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from sklearn.utils.class_weight import compute_class_weight

In [15]:
# Set random seeds for reproducibility
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

In [16]:
# Load tensors and convert to TensorFlow format
TENSOR_DIR = Path(r"C:\Users\omarl\Downloads\pfizer_tensors")

X_torch = torch.load(TENSOR_DIR / "X_features.pt")
y_torch = torch.load(TENSOR_DIR / "y_labels.pt")
fold_torch = torch.load(TENSOR_DIR / "folds.pt")

X_tf = tf.convert_to_tensor(X_torch.cpu().numpy(), dtype=tf.float32)
y_tf = tf.convert_to_tensor(y_torch.cpu().numpy(), dtype=tf.int64)
fold_tf = tf.convert_to_tensor(fold_torch.cpu().numpy(), dtype=tf.int64)

print("X:", X_tf.shape)
print("y:", y_tf.shape)
print("folds:", fold_tf.shape)

C:\Users\omarl\AppData\Local\Temp\ipykernel_12452\545063573.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  X_torch = torch.load(TENSOR_DIR / "X_features.pt")
C:\Users\o

X: (20931, 86, 65)
y: (20931,)
folds: (20931,)


In [17]:
# Filter out unlabeled samples (y == -1)
labeled_mask = y_tf != -1

X_labeled = tf.boolean_mask(X_tf, labeled_mask)
y_labeled = tf.boolean_mask(y_tf, labeled_mask)
fold_labeled = tf.boolean_mask(fold_tf, labeled_mask)

target_names = ["SEG_A", "SEG_B", "SEG_C"]

print("X labeled:", X_labeled.shape)
print("y labeled:", y_labeled.shape)
print(pd.Series(y_labeled.numpy()).value_counts().sort_index())

X labeled: (11899, 86, 65)
y labeled: (11899,)
0    6406
1    3349
2    2144
Name: count, dtype: int64


In [18]:
# Create train, validation, and test splits based on folds
test_fold = 3
val_fold = 4

train_mask = (fold_labeled != test_fold) & (fold_labeled != val_fold)
val_mask = fold_labeled == val_fold
test_mask = fold_labeled == test_fold

X_train_tensor = tf.boolean_mask(X_labeled, train_mask)
y_train = tf.boolean_mask(y_labeled, train_mask)

X_val_tensor = tf.boolean_mask(X_labeled, val_mask)
y_val = tf.boolean_mask(y_labeled, val_mask)

X_test_tensor = tf.boolean_mask(X_labeled, test_mask)
y_test = tf.boolean_mask(y_labeled, test_mask)

print("Train tensor:", X_train_tensor.shape, y_train.shape)
print("Val tensor:", X_val_tensor.shape, y_val.shape)
print("Test tensor:", X_test_tensor.shape, y_test.shape)

Train tensor: (7140, 86, 65) (7140,)
Val tensor: (2379, 86, 65) (2379,)
Test tensor: (2380, 86, 65) (2380,)


In [19]:
# Flatten the feature tensors for use in a dense neural network
X_train = tf.reshape(X_train_tensor, (X_train_tensor.shape[0], -1))
X_val = tf.reshape(X_val_tensor, (X_val_tensor.shape[0], -1))
X_test = tf.reshape(X_test_tensor, (X_test_tensor.shape[0], -1))

print("Train flat:", X_train.shape)
print("Val flat:", X_val.shape)
print("Test flat:", X_test.shape)

Train flat: (7140, 5590)
Val flat: (2379, 5590)
Test flat: (2380, 5590)


In [20]:
# Compute class weights for the binary classification
classes = np.unique(y_train.numpy())

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train.numpy()
)

class_weight_multiclass = dict(zip(classes, weights))
class_weight_multiclass

{0: 0.619146722164412, 1: 1.1846689895470384, 2: 1.8492618492618493}

In [21]:
# Function to build a dense neural network model with configurable hyperparameters
def build_dense_multiclass_model(
    input_dim,
    hidden_layers=(256, 128),
    activation="relu",
    dropout_rate=0.3,
    learning_rate=3e-4,
    l2_reg=1e-4,
    use_batch_norm=True
):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    
    model = models.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    
    if use_batch_norm:
        model.add(layers.BatchNormalization())
    
    for units in hidden_layers:
        model.add(layers.Dense(
            units,
            activation=activation,
            kernel_regularizer=regularizers.l2(l2_reg)
        ))
        
        if use_batch_norm:
            model.add(layers.BatchNormalization())
        
        model.add(layers.Dropout(dropout_rate))
    
    model.add(layers.Dense(3, activation="softmax"))
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    
    return model

In [22]:
# Define grid search hyperparameters
input_dim = X_train.shape[1]

architectures = {
    "small_1_layer": (128,),
    "medium_2_layers": (256, 128),
    "large_2_layers": (512, 256),
    "deep_3_layers": (512, 256, 128),
    "wide_3_layers": (512, 512, 256),
}

activations = ["relu", "gelu", "elu", "swish"]

dropout_rates = [0.2, 0.3, 0.4]

learning_rates = [1e-3, 5e-4, 3e-4]

In [23]:
# Function to evaluate model performance on the evaluation set
def evaluate_multiclass_model(model, X_eval, y_eval, target_names):
    proba = model.predict(X_eval, verbose=0)
    y_pred = np.argmax(proba, axis=1)
    y_true = y_eval.numpy()
    
    report = classification_report(
        y_true,
        y_pred,
        target_names=target_names,
        output_dict=True
    )
    
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
        "SEG_A_recall": report["SEG_A"]["recall"],
        "SEG_B_recall": report["SEG_B"]["recall"],
        "SEG_C_recall": report["SEG_C"]["recall"],
        "SEG_A_f1": report["SEG_A"]["f1-score"],
        "SEG_B_f1": report["SEG_B"]["f1-score"],
        "SEG_C_f1": report["SEG_C"]["f1-score"],
    }

In [24]:
# Perform grid search over the defined hyperparameters and store results
dense_grid_results = []

for arch_name, hidden_layers in architectures.items():
    for activation in activations:
        for dropout_rate in dropout_rates:
            for lr in learning_rates:
                
                print("\n==========================================")
                print(f"Architecture: {arch_name} | Layers: {hidden_layers}")
                print(f"Activation: {activation} | Dropout: {dropout_rate} | LR: {lr}")
                print("==========================================")
                
                model = build_dense_multiclass_model(
                    input_dim=input_dim,
                    hidden_layers=hidden_layers,
                    activation=activation,
                    dropout_rate=dropout_rate,
                    learning_rate=lr,
                    l2_reg=1e-4,
                    use_batch_norm=True
                )
                
                early_stop = callbacks.EarlyStopping(
                    monitor="val_loss",
                    patience=8,
                    restore_best_weights=True
                )
                
                reduce_lr = callbacks.ReduceLROnPlateau(
                    monitor="val_loss",
                    factor=0.5,
                    patience=4,
                    min_lr=1e-5
                )
                
                history = model.fit(
                    X_train,
                    y_train,
                    validation_data=(X_val, y_val),
                    epochs=60,
                    batch_size=128,
                    class_weight=class_weight_multiclass,
                    callbacks=[early_stop, reduce_lr],
                    verbose=0
                )
                
                val_metrics = evaluate_multiclass_model(
                    model,
                    X_val,
                    y_val,
                    target_names
                )
                
                result = {
                    "architecture_name": arch_name,
                    "hidden_layers": str(hidden_layers),
                    "activation": activation,
                    "dropout_rate": dropout_rate,
                    "learning_rate": lr,
                    "epochs_trained": len(history.history["loss"]),
                    "best_val_loss": min(history.history["val_loss"]),
                    "best_val_accuracy": max(history.history["val_accuracy"]),
                    **val_metrics
                }
                
                dense_grid_results.append(result)
                
                print(
                    f"Val Macro F1: {val_metrics['macro_f1']:.4f} | "
                    f"Weighted F1: {val_metrics['weighted_f1']:.4f} | "
                    f"SEG_C Recall: {val_metrics['SEG_C_recall']:.4f}"
                )


Architecture: small_1_layer | Layers: (128,)
Activation: relu | Dropout: 0.2 | LR: 0.001
Val Macro F1: 0.5330 | Weighted F1: 0.6055 | SEG_C Recall: 0.3925

Architecture: small_1_layer | Layers: (128,)
Activation: relu | Dropout: 0.2 | LR: 0.0005
Val Macro F1: 0.5174 | Weighted F1: 0.5943 | SEG_C Recall: 0.3154

Architecture: small_1_layer | Layers: (128,)
Activation: relu | Dropout: 0.2 | LR: 0.0003
Val Macro F1: 0.5316 | Weighted F1: 0.6060 | SEG_C Recall: 0.3458

Architecture: small_1_layer | Layers: (128,)
Activation: relu | Dropout: 0.3 | LR: 0.001
Val Macro F1: 0.5430 | Weighted F1: 0.6138 | SEG_C Recall: 0.4159

Architecture: small_1_layer | Layers: (128,)
Activation: relu | Dropout: 0.3 | LR: 0.0005
Val Macro F1: 0.5210 | Weighted F1: 0.6020 | SEG_C Recall: 0.3178

Architecture: small_1_layer | Layers: (128,)
Activation: relu | Dropout: 0.3 | LR: 0.0003
Val Macro F1: 0.5305 | Weighted F1: 0.6075 | SEG_C Recall: 0.3364

Architecture: small_1_layer | Layers: (128,)
Activation: re

In [25]:
# Convert results to DataFrame and display top performing configurations
dense_grid_df = pd.DataFrame(dense_grid_results)

dense_grid_df.sort_values(
    ["macro_f1", "weighted_f1", "SEG_C_recall"],
    ascending=False
).head(10)

,architecture_name,hidden_layers,activation,dropout_rate,learning_rate,epochs_trained,best_val_loss,best_val_accuracy,accuracy,macro_f1,weighted_f1,SEG_A_recall,SEG_B_recall,SEG_C_recall,SEG_A_f1,SEG_B_f1,SEG_C_f1
161,wide_3_layers,"(512, 512, 256)",gelu,0.4,0.0003,12,1.021786,0.641026,0.641026,0.567214,0.636048,0.800156,0.486567,0.406542,0.774462,0.513386,0.413793
141,deep_3_layers,"(512, 256, 128)",swish,0.4,0.0010,12,0.994255,0.635561,0.635561,0.566262,0.634427,0.761905,0.561194,0.373832,0.763096,0.542569,0.393120
79,large_2_layers,"(512, 256)",relu,0.4,0.0005,11,1.020470,0.626314,0.626314,0.565644,0.628678,0.758782,0.432836,0.532710,0.764451,0.484545,0.447937
104,large_2_layers,"(512, 256)",swish,0.3,0.0003,11,1.011871,0.625893,0.625893,0.565258,0.626525,0.757221,0.464179,0.485981,0.756335,0.494043,0.445396
107,large_2_layers,"(512, 256)",swish,0.4,0.0003,11,1.023414,0.623371,0.623371,0.563227,0.624109,0.752537,0.471642,0.474299,0.753714,0.490303,0.445664
121,deep_3_layers,"(512, 256, 128)",gelu,0.3,0.0005,11,0.985857,0.630517,0.630517,0.561102,0.629640,0.771272,0.504478,0.406542,0.764706,0.517215,0.401384
116,deep_3_layers,"(512, 256, 128)",relu,0.4,0.0003,13,0.984987,0.641446,0.637243,0.559963,0.630460,0.793130,0.520896,0.352804,0.766214,0.525998,0.387677
142,deep_3_layers,"(512, 256, 128)",swish,0.4,0.0005,12,0.996575,0.635561,0.635561,0.559826,0.631003,0.795472,0.486567,0.390187,0.771970,0.511774,0.395735
173,wide_3_layers,"(512, 512, 256)",swish,0.2,0.0003,10,1.058046,0.627995,0.627995,0.557678,0.625815,0.775956,0.486567,0.406542,0.765794,0.493939,0.413302
118,deep_3_layers,"(512, 256, 128)",gelu,0.2,0.0005,11,1.015588,0.619588,0.619588,0.557390,0.622527,0.736924,0.523881,0.418224,0.750099,0.518464,0.403608
